## Time-varying EWMA Baseline

We checked whether the data itself shows a source of variation to adapt to using EWMA: **intraday non-stationarity**. Does 'normal' actually drift over the course of the day? If so, a baseline that adapts over time (via EWMA) addresses something real in the data and will lead to more accurate predictions.


In [1]:
import pandas as pd
import numpy as np

file_path = '../data/historical.csv'
data = pd.read_csv(file_path)


### Hourly breakdown

Grouping transactions by hour of day (ignoring the fact that there are 2 days) and computing, per hour: average `Amount` and fraud rate (Amount calculated on **non-fraud** transactions, we are looking for a baseline of what normal spending amounts are), and the mean/std of `V1`-`V28` on **non-fraud** transactions only, since that's the population the baseline from notebook 2 is fit on.

In [2]:
amount_averages = []
fraud_rates = []
feature_means = []
feature_stds = []

for i in range(24):
    hour = data.loc[(data.Time%86400) // 3600 == i]
    non_fraud_hour = hour.loc[data.Class == 0]


    amount_per_hour = non_fraud_hour.loc[:, 'Amount'].to_numpy().mean()
    amount_averages.append((float(round(amount_per_hour, 3)), i))

    frauds = hour.loc[:, 'Class'].to_numpy().mean()
    fraud_rates.append((float(round(frauds,5)), i))

    features = non_fraud_hour.loc[:, ['V' + str(i) for i in range(1, 29, 1)]].to_numpy()
    feature_means.append(features.mean(axis=0))
    feature_stds.append(features.std(axis=0, ddof=1))



print(amount_averages)
print(fraud_rates)
print(np.array(feature_means).shape)
print(np.array(feature_stds).shape)


[(60.084, 0), (61.636, 1), (66.464, 2), (51.193, 3), (82.959, 4), (48.385, 5), (67.92, 6), (65.731, 7), (90.909, 8), (104.13, 9), (108.626, 10), (111.349, 11), (106.754, 12), (99.7, 13), (102.191, 14), (100.446, 15), (101.171, 16), (98.48, 17), (79.105, 18), (77.544, 19), (75.634, 20), (72.008, 21), (70.263, 22), (68.362, 23)]
[(0.00093, 0), (0.00241, 1), (0.01613, 2), (0.005, 3), (0.01187, 4), (0.00338, 5), (0.00142, 6), (0.00372, 7), (0.00084, 8), (0.00045, 9), (0.00051, 10), (0.0033, 11), (0.00102, 12), (0.00084, 13), (0.00146, 14), (0.00147, 15), (0.00147, 16), (0.00203, 17), (0.00193, 18), (0.00128, 19), (0.00111, 20), (0.00105, 21), (0.00055, 22), (0.00197, 23)]
(24, 28)
(24, 28)


**Finding:** both `Amount` and fraud rate vary substantially by hour. Fraud rate ranges from about 0.045% (hour 9) to 1.6% (hour 2), roughly a 35x difference, with several overnight hours running well above the ~0.17% overall average. Average `Amount` ranges from about $48 (hour 5) to $111 (hour 11), tracking the expected day/night transaction volume pattern. This alone doesn't prove the `V` feature baseline itself drifts, but it's a first signal that "normal" isn't static across the day.

### Non-hourly reference statistics

To judge whether an hour's mean/std is meaningfully different, we need something to compare it against: the mean and std of `V1`-`V28` across the *entire* non-fraud historical set, ignoring hour entirely. 

In [3]:
nonfraud_mean = data.loc[data.Class == 0].loc[:, ['V' + str(i) for i in range(1, 29, 1)]].to_numpy().mean(axis=0)
nonfraud_std = data.loc[data.Class == 0].loc[:, ['V' + str(i) for i in range(1, 29, 1)]].to_numpy().std(axis=0, ddof=1)

print(nonfraud_mean.shape)
print(nonfraud_std.shape)

(28,)
(28,)


### Standardized mean drift

We standardize each hour's mean against the overall mean and overall std (`(hour_mean - overall_mean) / overall_std`), puting every feature on the same scale ("how many standard deviations off is this hour's typical value"). Taking the max absolute value across hours, per feature, ranks which features drift the most.

In [4]:
max_difference = np.max(np.abs((feature_means - nonfraud_mean)/nonfraud_std), axis=0).tolist()
hour_occurance = np.argmax(np.abs((feature_means - nonfraud_mean)/nonfraud_std), axis=0).tolist()
features = ['V' + str(i) for i in range(1, 29, 1)]
mean_deviations = sorted(list(zip(max_difference, hour_occurance, features)), reverse=True)

# prints as (# of standard deviations from mean, hour of occurance, feature #)
print(mean_deviations)

[(2.485529374783798, 6, 'V12'), (1.5917998307177232, 1, 'V13'), (1.473931705602061, 5, 'V14'), (1.471313998875495, 5, 'V9'), (0.9848074312177134, 4, 'V11'), (0.9223249652221016, 2, 'V15'), (0.8550005530190069, 1, 'V17'), (0.6267770709175738, 5, 'V26'), (0.56760700644466, 5, 'V10'), (0.37740527374923727, 5, 'V3'), (0.3460748252971494, 4, 'V4'), (0.3429350357888838, 3, 'V18'), (0.3086316692520353, 1, 'V22'), (0.26659899271162024, 4, 'V19'), (0.23500995484122034, 2, 'V5'), (0.22494361295603468, 0, 'V2'), (0.21832739924543212, 4, 'V6'), (0.21486127731638316, 0, 'V7'), (0.19493811592321766, 3, 'V16'), (0.19240666111306565, 2, 'V8'), (0.18947213559015486, 4, 'V25'), (0.13262287637353926, 0, 'V27'), (0.13062399343628678, 5, 'V21'), (0.1233439801628787, 3, 'V20'), (0.11679184022504, 2, 'V1'), (0.09771654207018717, 5, 'V24'), (0.06413291930389892, 4, 'V23'), (0.06299396784130913, 5, 'V28')]


**Finding:** `V12`, `V13`, `V14`, and `V9` show the largest standardized mean drift (roughly 1.5–2.5 SDs off the overall mean at their most extreme hour), while several other features (`V23`, `V28`, `V24`, `V1`) barely move at all (well under 0.2 SDs). The drift isn't uniform across features, a handful of features carry most of the volatility, which is useful to know: an adaptive baseline mostly needs to be correcting for a specific subset of features, not all 28 uniformly.

### Standardized spread (std) drift

Mean drift alone doesn't capture whether a feature's *spread* also changes by hour — which matters directly for Mahalanobis distance, since the covariance (not just the mean) is part of the baseline. Comparing each hour's std to the overall std, both the widest ratio (spread gets wider than usual) and the narrowest ratio (spread gets tighter than usual) are tracked, since both directions are equally informative about non-stationarity.

In [5]:
widest_dev = np.max((feature_stds / nonfraud_std), axis=0).tolist()
hour_widest = np.argmax((feature_stds / nonfraud_std), axis=0).tolist()

narrowest_dev = np.min((feature_stds / nonfraud_std), axis=0).tolist()
hour_narrowest = np.argmin((feature_stds / nonfraud_std), axis=0).tolist()

wider = sorted(list(zip(widest_dev, hour_widest, features)), reverse=True)
narrower = sorted(list(zip(narrowest_dev, hour_narrowest, features)), reverse=False)

print(wider)
print(narrower)




[(1.7016276546073574, 2, 'V21'), (1.5790930230763747, 6, 'V17'), (1.4406455791940185, 15, 'V28'), (1.3658860205813488, 2, 'V20'), (1.31165061563547, 6, 'V3'), (1.2996968608093853, 23, 'V8'), (1.2821620096050108, 15, 'V23'), (1.2805161192878038, 2, 'V10'), (1.2767723352330693, 22, 'V7'), (1.186846520591407, 2, 'V2'), (1.1809811241755475, 2, 'V27'), (1.1729161612226269, 4, 'V26'), (1.171334895328759, 22, 'V5'), (1.1603016758299693, 15, 'V1'), (1.1452237424586291, 6, 'V14'), (1.1406815115107503, 0, 'V4'), (1.1174891405362277, 2, 'V15'), (1.1026805822004067, 2, 'V22'), (1.074666494579343, 5, 'V19'), (1.0730982906075546, 22, 'V6'), (1.0646346724911142, 9, 'V16'), (1.0555747517181728, 2, 'V9'), (1.048182845574215, 10, 'V18'), (1.0430916308879221, 16, 'V25'), (1.02518798839781, 12, 'V24'), (1.0219372200539278, 4, 'V11'), (0.9664096754702043, 5, 'V12'), (0.9420867598833371, 1, 'V13')]
[(0.6693314531001414, 7, 'V28'), (0.6765359449813615, 3, 'V12'), (0.6800133971923513, 1, 'V23'), (0.7417947030

**Finding:** `V21`, `V17`, `V28`, and `V20` show the widest spread-widening (up to ~1.7x the overall std at their peak hour), while `V28`, `V12`, and `V23` show the most spread-tightening (down to ~0.67–0.68x the overall std at their tightest hour). Spread also varies by hour, which the EWMA needs to account for.

### Conclusion

Both the mean and the spread of the non-fraud baseline drift meaningfully by hour of day. This justifies building the EWMA-adapted baseline to combat variation in 'normal' throughout the day.
